<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>ACE-Step 1.5 AI Music Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>LoRA Training Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 20px 0;'>Free on Google Colab T4 GPU | MIT License</p>

   ---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />

 <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---


### About This Notebook

This notebook uses the **full ACE-Step 1.5 Gradio interface** with all advanced features:

| Feature | Description |
|----|----|  
| **LoRA Training & Loading** | Train custom models on your music style |
| **Advanced Controls** | BPM, key scale, time signature, duration |
| **Multiple Modes** | Text2Music, Cover, Repaint |
| **Batch Processing** | Generate multiple variations |
| **Model Management** | Load custom checkpoints |

### Before You Start
> Go to **Runtime -> Change runtime type -> T4 GPU**


In [ ]:
# @title Step 1: GPU Check & PyTorch Verification
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    # Only reinstall if CUDA doesn't work properly
    try:
        x = torch.zeros(1, device="cuda")
        del x
        print("\n✅ PyTorch CUDA is working!")
    except Exception as e:
        print(f"\n⚠️ CUDA test failed: {e}")
        print("Reinstalling PyTorch with CUDA 12.1...")
        !pip install --force-reinstall --quiet torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | grep -v "WARNING\|must restart"
        print("✅ PyTorch reinstalled")
else:
    print("\n❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")


In [ ]:
# @title Step 2: Install uv, Clone repo, Install dependencies
import os

# Install uv package manager
print("📦 Installing uv package manager...")
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>&1 | tail -1
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

# Clone or update repository
%cd /content
if os.path.exists("/content/ACE-Step-1.5"):
    print("\n📂 Repository exists, pulling latest...")
    %cd /content/ACE-Step-1.5
    !git pull
else:
    print("\n📥 Cloning ACE-Step 1.5...")
    !git clone https://github.com/ACE-Step/ACE-Step-1.5.git
    %cd /content/ACE-Step-1.5

# Install all dependencies
print("\n⚙️ Installing dependencies (2-3 minutes)...")
!uv sync 2>&1 | tail -5

print("\n✅ All dependencies installed!")


In [ ]:
# @title Step 2: Run Gradio UI

import os, re

os.chdir("/content/ACE-Step-1.5")

# ── 0. Reset previous patches ──
!git checkout -- acestep/ui/gradio/events/generation/service_init.py acestep/llm_inference.py acestep/gpu_config.py acestep/ui/gradio/interfaces/__init__.py 2>/dev/null

# ── 1. Download 0.6B LM model ──
print("📥 Downloading 0.6B LM model (~1.4GB)...")
!uv run acestep-download --model acestep-5Hz-lm-0.6B
print("✅ Download complete")
!ls -la checkpoints/acestep-5Hz-lm-0.6B/ 2>/dev/null || echo "❌ 0.6B still not found!"

# ── 2. branding — replace header HTML ──
init_file = "acestep/ui/gradio/interfaces/__init__.py"
with open(init_file, "r") as f:
    code = f.read()

if "AIQUEST" not in code:
    # Find the main-header div and replace it
    old_div_start = code.find('<div class="main-header">')
    old_div_end = code.find('</div>', old_div_start) + len('</div>')

    if old_div_start >= 0:
        old_html = code[old_div_start:old_div_end]
        new_html = '''<div style="text-align:center; padding:25px; background:linear-gradient(135deg,#667eea 0%,#764ba2 100%); border-radius:12px; box-shadow:0 8px 16px rgba(0,0,0,0.2); margin-bottom:20px;">
            <h1 style="font-size:2.2em; margin:0; color:white;">🎵 ACE-Step 1.5 - AI Music Generator</h1>
            <p style="font-size:1.1em; color:rgba(255,255,255,0.9); margin:6px 0 12px;">LoRA Training & Loading | Advanced Controls | Model Management</p>
            <div style="display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">
                <a href="https://www.youtube.com/@AiQuestacademy" target="_blank"
                   style="background:#FF0000; color:#fff; padding:8px 18px; border-radius:8px;
                    text-decoration:none; font-weight:600; font-size:0.95em;">
                   ▶ YouTube - AIQUEST</a>
                <a href="https://x.com/AiQuestacademy" target="_blank"
                   style="background:#0000; color:#fff; padding:8px 18px; border-radius:8px;
                    text-decoration:none; font-weight:600; font-size:0.95em;">
                   𝕏 Follow on X</a>
            </div>
            <p style="margin-top:10px; font-size:0.85em; color:rgba(255,255,255,0.7);">
                Notebook by <b>AIQUEST</b> - Subscribe for more AI tools & tutorials!</p>
        </div>'''

        code = code.replace(old_html, new_html)
        # Remove f-string since we no longer use {t()} calls in the header
        code = code.replace('gr.HTML(f"""', 'gr.HTML("""', 1)
        with open(init_file, "w") as f:
            f.write(code)

    else:
        print("⚠️ Could not find main-header div")

# ── 3. Patch service_init.py: force LLM init ONLY on first call ──
si_file = "acestep/ui/gradio/events/generation/service_init.py"
with open(si_file, "r") as f:
    si_src = f.read()

patch_marker = "# AIQUEST_FORCE_LLM_PATCH"
if patch_marker not in si_src:
    old = '    quant_value = "int8_weight_only" if quantization else None'
    new = f'''    # {patch_marker}
    if not hasattr(init_service_wrapper, '_aiquest_first_done'):
        # First call: force LLM init with 0.6B
        init_llm = True
        if not lm_model_path or "1.7B" in str(lm_model_path):
            lm_model_path = "acestep-5Hz-lm-0.6B"
        init_service_wrapper._aiquest_first_done = True
        print(f"🔧 [AIQUEST] First init: forced init_llm=True, lm_model_path={{lm_model_path}}")
    else:
        # Subsequent calls: respect UI selection
        print(f"🔧 [AIQUEST] UI-driven init: init_llm={{init_llm}}, lm_model_path={{lm_model_path}}")

    quant_value = "int8_weight_only" if quantization else None'''
    si_src = si_src.replace(old, new)
    with open(si_file, "w") as f:
        f.write(si_src)
    print("✅ service_init.py patched: force 0.6B on first init only")
else:
    print("✅ service_init.py already patched")

# ── 4. Patch llm_inference.py: enforce_eager for T4 ──
llm_file = "acestep/llm_inference.py"
with open(llm_file, "r") as f:
    llm_src = f.read()
llm_src = re.sub(
    r'enforce_eager_for_vllm\s*=\s*bool\(is_rocm\)',
    'enforce_eager_for_vllm = True  # Force eager for T4',
    llm_src
)
with open(llm_file, "w") as f:
    f.write(llm_src)
print("✅ llm_inference.py patched: enforce_eager=True")

# ── 5. Clear __pycache__ so Python reads patched .py files ──
!find /content/ACE-Step-1.5/acestep -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null
print("✅ Cleared all __pycache__")

# ── 6. Launch ──
print("\n🚀 Launching ACE-Step 1.5 with LoRA Training UI...")
print("🔗 Watch for the public Gradio URL!\n")

!MPLBACKEND=agg uv run acestep --share --lm_model_path acestep-5Hz-lm-0.6B --offload_to_cpu true

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⭐ star the repo** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/▶%20Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20𝕏-0000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

**Made with ❤️ by AIQUEST** | [@aiquestacademy](https://youtube.com/@aiquestacademy)

</div>
